In [1]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import create_preprocessor

FE_DF_PATH = Path("../data/processed/feature_engineered_df.parquet")
METRICS_DF_PATH = Path("../data/processed/metrics_df.csv")

fe_df = pd.read_parquet(FE_DF_PATH)
metrics_df = pd.read_csv(METRICS_DF_PATH)

In [2]:
# select features
model_df = fe_df[["Store", "Customers", "Day", "WeekOfYear", "Quarter", "Open", "Season", 
           "Promo", "SchoolHoliday", "CompetitionDistance", "CompetitionAgeMonths", 
           "WasPromo2Active", "StoreType", "Year", "Month", 
           "DayOfWeek", "StateHoliday", "Assortment", "PromoInterval", "Sales"]]

# train and test sets
train = model_df[(model_df.Year < 2015) | ((model_df.Year == 2015) & (model_df.Month < 6))]
valid = model_df[(model_df.Year == 2015) & (model_df.Month >= 6)]

X_train = train.drop(columns="Sales")
y_train = train.Sales

X_valid = valid.drop(columns="Sales")
y_valid = valid.Sales

In [3]:
# select numerical and categorical features
num_features = X_train.select_dtypes(include="number").columns
cat_features = X_train.select_dtypes(exclude="number").columns

preprocessor = create_preprocessor(num_features, cat_features)

# create the pipeline
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

In [4]:
# tune hyperparameters
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit

param_grid = {
    'regressor__max_depth': [None, 10, 20],
    'regressor__n_estimators': [100, 300],
    'regressor__min_samples_leaf': [1, 2],
    'regressor__min_samples_split': [2, 5]
}

tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(pipeline, param_grid, cv=tscv, scoring="r2")
model_grid = grid.fit(X_train, y_train)

# print best parameters and best score
print(f"Best hyperparameters: {model_grid.best_params_}")
print(f"Best CV R²: {model_grid.best_score_:.4f}")

Best hyperparameters: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 300}
Best CV R²: 0.9787


In [5]:
# calculate metrics and save the results
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

y_pred = model_grid.predict(X_valid)

model_grid_metrics_df = pd.DataFrame([{
    "Model": "Random Forest (Tuned)",
    "CV Mean R²": model_grid.best_score_,
    "CV Std": model_grid.cv_results_["std_test_score"][model_grid.best_index_],
    "Test R² score": r2_score(y_valid, y_pred),
    "MAE": mean_absolute_error(y_valid, y_pred),
    "MSE": mean_squared_error(y_valid, y_pred),
    "MAPE": mean_absolute_percentage_error(y_valid, y_pred),
}]).round(2)

# concat it
final_metrics_df = pd.concat([metrics_df, model_grid_metrics_df], ignore_index=True)
# save it
final_metrics_df.to_csv("../data/processed/final_metrics_df.csv", index=False)

In [6]:
# save the cv results
cv_results = pd.DataFrame(model_grid.cv_results_)
cv_results.to_csv("../data/processed/random_forest_gridsearch_cv_results.csv", index=False)

In [7]:
# save the best model
import joblib
joblib.dump(model_grid.best_estimator_, "../outputs/models/random_forest_tuned.joblib")

['../outputs/models/random_forest_tuned.joblib']